# 05 · Explainability Lab

**Purpose.** Проверить рассогласованность объяснимости: top_drivers по PC1 против module_contributions по всем PC (EVR-weighted).

**What to look for.**
- доля строк, где доминирующий модуль PC1 != EVR-attribution
- сколько дисперсии объясняет PC1 (мало → PC1 не репрезентативен)
- распределение вкладов модулей
- драйверы на ключевых датах (макс эпизодов, последняя дата)

> Это lab-ноутбук: выводы здесь предварительные, не финальный отчёт. Меняй параметры в ячейке *Parameters* и перезапускай.

In [ ]:
# --- bootstrap: запуск из корня проекта (рядом с data/ и backend/) ---
import sys, os
from pathlib import Path
# найти корень проекта и встать в него
_here = Path.cwd()
_root = next((p for p in [_here, *_here.parents] if (p / 'data' / 'processed').is_dir()), _here)
os.chdir(_root)
sys.path.insert(0, str(_root))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 60)
from lab import utils as u
print('project root:', _root.name)

## Parameters
Меняй значения здесь и перезапускай ноутбук.

In [ ]:
TOP_N = 3
EXCLUDE = []   # попробуй ['m1_flag_end_of_period','m1_signal_final']

In [ ]:
d = u.load_final_dataset()
av = [f for f in u.available_whitelist(d) if f not in EXCLUDE]
art = u.fit_lsi_like_model(d, av)
pca = art['pca']; feats = art['features']; Xs = art['scaled_matrix']
print('PC1 explains', round(pca.explained_variance_ratio_[0]*100,2),'% of variance')
print('cum 10 PC:', round(pca.explained_variance_ratio_.sum()*100,2),'%')

### Agreement: PC1-drivers vs EVR-weighted attribution

In [ ]:
u.driver_agreement_rate(art)

### Распределение доминирующего модуля: PC1 vs EVR

In [ ]:
mc = u.module_contributions(Xs, pca, feats)
evr_dom = mc.idxmax(axis=1).str.lower()
pc1 = pca.components_[0]
pc1_dom = pd.Series([feats[int(np.argmax(np.abs(r*pc1)))].split('_',1)[0] for r in Xs])
cmp = pd.DataFrame({'PC1_dominant': pc1_dom.value_counts(), 'EVR_dominant': evr_dom.value_counts()}).fillna(0).astype(int)
cmp

### Средние вклады модулей (EVR-attribution)

In [ ]:
mc.mean().round(2).sort_values(ascending=False)

### PCA loadings и structural_weights

In [ ]:
load = pd.DataFrame(pca.components_[:5].T, index=feats, columns=[f'PC{i+1}' for i in range(5)]).round(3)
load['structural_weight'] = u.structural_weights(pca).round(4)
load.sort_values('structural_weight', ascending=False)

### Драйверы на ключевых датах: два метода рядом

In [ ]:
key_dates = {}
dd = pd.DataFrame({'date': art['date'], 'lsi': art['lsi']})
for k,(a,b) in u.STRESS_EPISODES.items():
    seg = dd[(dd['date']>=a)&(dd['date']<=b)]
    if len(seg):
        key_dates[k] = seg.loc[seg['lsi'].idxmax(),'date']
key_dates['latest'] = dd['date'].iloc[-1]
rows = []
for label, dt in key_dates.items():
    i = int(np.where(art['date']==np.datetime64(pd.Timestamp(dt)))[0][0])
    rows.append({'label':label,'date':pd.Timestamp(dt).date(),'lsi':round(float(art['lsi'][i]),2),
                 'PC1_drivers': u.pc1_drivers(Xs[i], pca, feats, TOP_N),
                 'EVR_drivers': u.evr_weighted_drivers(Xs[i], pca, feats, TOP_N)})
pd.DataFrame(rows)

> **Почему это не SHAP / не causal importance.** Оба метода — PCA-эвристики: PC1-drivers берут только первую компоненту (~19% дисперсии), attribution взвешивает |loadings| по EVR. Ни тот, ни другой не учитывает взаимодействие признаков и не является каузальной декомпозицией вклада в IsolationForest-score.

## Notes / Open questions

- Agreement ~60% означает: UI может показывать разный 'главный модуль' в драйверах и в пироге вкладов.
- Стоит выбрать ОДИН метод (например, EVR-attribution для обоих) — проверь, как меняются драйверы.
- Для честной объяснимости можно сравнить с permutation importance на anomaly score (TODO в 07).